# Mamba-2 Multi-Model Head Heterogeneity 분석

## 목표
- 130m → 2.7b 모델 스케일별로 head 이질성 패턴 비교
- A_disc vs effective rank scatter (Type A/B/C 분류)
- slow head (Type A)만의 rank saturation curve
- 모델 크기에 따라 long-term memory head 비율이 달라지는가?

## 분석 대상
```
state-spaces/mamba2-130m
state-spaces/mamba2-370m
state-spaces/mamba2-780m
state-spaces/mamba2-1.3b
state-spaces/mamba2-2.7b
```

## 0. 설치

In [1]:
import sys, subprocess, os

# 1. causal-conv1d
subprocess.run([sys.executable, '-m', 'pip', 'install',
              'git+https://github.com/Dao-AILab/causal-conv1d.git',
              '--no-build-isolation'], check=True)

# 2. mamba
os.chdir('/root')
subprocess.run(['rm', '-rf', 'mamba'], check=False)
subprocess.run(['git', 'clone', 'https://github.com/state-spaces/mamba.git'],
check=True)
os.chdir('/root/mamba')

env = os.environ.copy()
env.update({'CAUSAL_CONV1D_FORCE_BUILD': 'TRUE',
          'CAUSAL_CONV1D_SKIP_CUDA_BUILD': 'TRUE',
          'CAUSAL_CONV1D_FORCE_CXX11_ABI': 'TRUE'})

subprocess.run([sys.executable, '-m', 'pip', 'install',
'--no-build-isolation', '.'],
             env=env, check=True)
print("✅ 완료")

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/dill-0.3.9-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_thunder-0.2.2.dev0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.14.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1.3.0-py3.12.egg is deprecated. pip 25.1 w

  Cloning https://github.com/Dao-AILab/causal-conv1d.git to /tmp/pip-req-build-57s5_xxx
  Resolved https://github.com/Dao-AILab/causal-conv1d.git to commit 0d2252d161f25f7907bbd05985c08ab98d255874
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for causal_conv1d: filename=causal_conv1d-1.6.1-cp312-cp312-linux_x86_64.whl size=170907398 sha256=ecda942d5e63c4b23663c62627ed1ea785de823cb266d01292b39c6fb18024d2
  Stored in directory: /tmp/pip-ephem-wheel-cache-bhwuhq87/wheels/8e/e0/3b/37dcbdd878a1a53bd01985b0e88cf2bd36bbd5246408162cbf
Successfully built causal_conv1d


Cloning into 'mamba'...
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/dill-0.3.9-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_thunder-0.2.2.dev0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.14.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1.3.0-py3.12.egg i

Processing /root/mamba
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 194.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 127.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 132.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.3/637.3 kB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 MB 137.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 137.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0

## 1. Import

In [1]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json
import os
from tqdm import tqdm
from transformers import AutoTokenizer
from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel
from mamba_ssm.utils.generation import InferenceParams
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


## 2. 실험 설정

In [2]:
MODEL_LIST = [
    # "state-spaces/mamba2-130m",
    # "state-spaces/mamba2-370m",
    "state-spaces/mamba2-780m",
    "state-spaces/mamba2-1.3b",
    # "state-spaces/mamba2-2.7b",
]

# VRAM 부족하면 작은 모델부터 주석 해제하며 실행
# MODEL_LIST = ["state-spaces/mamba2-130m", "state-spaces/mamba2-370m"]

# T range: 130m에서 검증된 포화 구간 포함
T_RANGE    = [32, 64, 128, 192, 256, 320, 384, 512, 768, 1024]
T_MAX      = max(T_RANGE)
N_SAMPLES  = 20   # 모델이 크면 시간 걸림 → 20개로 통일

# A_disc threshold for head type classification
SLOW_THRESHOLD = 0.99   # Type A: A_disc > 0.99
FAST_THRESHOLD = 0.50   # Type C: A_disc < 0.50
# Type B: 0.50 <= A_disc <= 0.99

SAVE_DIR = "./multi_model_results"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"T_RANGE  : {T_RANGE}")
print(f"N_SAMPLES: {N_SAMPLES}")
print(f"Save dir : {SAVE_DIR}")

T_RANGE  : [32, 64, 128, 192, 256, 320, 384, 512, 768, 1024]
N_SAMPLES: 20
Save dir : ./multi_model_results


## 3. 핵심 함수 정의

In [3]:
# ─── 샘플 텍스트 (네트워크 없이 사용) ───────────────────────────────────────
SAMPLE_TEXTS = [
    """The history of artificial intelligence began in antiquity, with myths, stories and rumors of artificial beings endowed with intelligence or consciousness by master craftsmen. The seeds of modern AI were planted by classical philosophers who attempted to describe the process of human thinking as the mechanical manipulation of symbols. This work culminated in the invention of the programmable digital computer in the 1940s, a machine based on the abstract essence of mathematical reasoning. This device and the ideas behind it inspired a handful of scientists to begin seriously discussing the possibility of building an electronic brain. The field of AI research was founded at a workshop held on the campus of Dartmouth College, USA during the summer of 1956. Those who attended would become the leaders of AI research for decades. Many of them predicted that a machine as intelligent as a human being would exist in no more than a generation and they were given millions of dollars to make this vision come true.""",
    """Machine learning is a method of data analysis that automates analytical model building. It is based on the idea that systems can learn from data, identify patterns and make decisions with minimal human intervention. Machine learning algorithms are trained on data sets containing examples of correct answers, then the algorithms learn to produce correct answers on their own. Deep learning is part of a broader family of machine learning methods based on artificial neural networks with representation learning. Learning can be supervised, semi-supervised or unsupervised. Deep learning architectures such as deep neural networks, recurrent neural networks, convolutional neural networks and transformers have been applied to fields including computer vision, speech recognition, natural language processing, machine translation, bioinformatics, drug design, medical image analysis, climate science, material inspection and board game programs.""",
    """The transformer architecture was introduced in the paper Attention Is All You Need by Vaswani et al. in 2017. It has become the dominant architecture for natural language processing tasks. The key innovation of the transformer is the self-attention mechanism, which allows the model to weigh the importance of different words in the input when producing each word of the output. Unlike recurrent neural networks, transformers process all tokens in parallel during training, making them much more efficient to train on modern hardware. The transformer has been scaled to billions of parameters, leading to large language models such as GPT, BERT, and T5. These models have shown remarkable performance on a wide range of natural language understanding and generation tasks, often matching or exceeding human performance on standardized benchmarks.""",
    """State space models represent a class of sequence models that have gained significant attention as alternatives to transformer architectures. Unlike transformers that use attention mechanisms with quadratic complexity, state space models process sequences with linear complexity. The Mamba architecture introduced selective state spaces, allowing the model parameters to be functions of the input, addressing the weakness of earlier state space models with discrete modalities. This selectivity allows Mamba to selectively propagate or forget information along the sequence length depending on the current token. Mamba achieves five times higher inference throughput than transformers while maintaining competitive performance on language modeling benchmarks. The architecture has been applied to various domains including language, audio, and genomics with promising results.""",
    """The retrieval augmented generation framework combines the strengths of large language models with information retrieval systems. In a standard RAG pipeline, a query is used to retrieve relevant documents from a knowledge base, and these documents are then provided as context to a language model to generate an answer. This approach allows language models to access up-to-date information without retraining and reduces hallucination by grounding responses in retrieved evidence. Vector databases store document embeddings and enable efficient similarity search to find relevant passages. The quality of retrieval significantly impacts the quality of generation, making the design of retrieval systems a critical component of RAG pipelines. Various improvements to basic RAG have been proposed, including multi-hop retrieval, reranking, and query expansion techniques.""",
    """Natural language processing is a subfield of linguistics, computer science, and artificial intelligence concerned with the interactions between computers and human language, in particular how to program computers to process and analyze large amounts of natural language data. The goal is a computer capable of understanding the contents of documents, including the contextual nuances of the language within them. The technology can then accurately extract information and insights contained in the documents, as well as categorize and organize the documents themselves. Challenges in natural language processing frequently involve speech recognition, natural language understanding, and natural language generation. Natural language processing has its roots in the 1950s. Already in 1950, Alan Turing published an article titled Computing Machinery and Intelligence which proposed what is now called the Turing test as a criterion of intelligence.""",
    """Reinforcement learning is an area of machine learning concerned with how intelligent agents ought to take actions in an environment in order to maximize the notion of cumulative reward. Reinforcement learning is one of three basic machine learning paradigms, alongside supervised learning and unsupervised learning. Reinforcement learning differs from supervised learning in not needing labelled input and output pairs to be presented, and in not needing sub-optimal actions to be explicitly corrected. Instead the focus is on finding a balance between exploration of uncharted territory and exploitation of current knowledge. The environment is typically stated in the form of a Markov decision process, because many reinforcement learning algorithms for this context utilize dynamic programming techniques.""",
    """Computer vision is an interdisciplinary scientific field that deals with how computers can gain high-level understanding from digital images or videos. From the perspective of engineering, it seeks to understand and automate tasks that the human visual system can do. Computer vision tasks include methods for acquiring, processing, analyzing and understanding digital images, and extraction of high-dimensional data from the real world in order to produce numerical or symbolic information, e.g. in the forms of decisions. Understanding in this context means the transformation of visual images into descriptions of the world that make sense to thought processes and can elicit appropriate action. This understanding is accomplished by unraveling the contextual information embedded in visual data.""",
]


def get_passages(n_samples, min_tokens, tokenizer):
    """샘플 텍스트를 반복해서 n_samples개의 passage 생성."""
    passages = []
    texts    = (SAMPLE_TEXTS * (n_samples // len(SAMPLE_TEXTS) + 1))[:n_samples * 2]
    for text in texts:
        ids = tokenizer.encode(text, return_tensors='pt')
        while ids.shape[1] < min_tokens:
            ids = torch.cat([ids, ids], dim=1)
        passages.append(ids[:, :min_tokens])
        if len(passages) >= n_samples:
            break
    return passages


def get_random_passages(n_samples, min_tokens, vocab_size=50277):
    return [torch.randint(0, vocab_size, (1, min_tokens)) for _ in range(n_samples)]

In [4]:
# ─── Effective Rank ──────────────────────────────────────────────────────────
def effective_rank(matrix: torch.Tensor, eps: float = 1e-9) -> float:
    S = torch.linalg.svdvals(matrix.float())
    S = S[S > eps]
    if len(S) == 0:
        return 1.0
    p       = S / S.sum()
    entropy = -(p * torch.log(p + eps)).sum()
    return torch.exp(entropy).item()


# ─── SSM State 추출 ──────────────────────────────────────────────────────────
def get_ssm_states(model, input_ids, device):
    """InferenceParams로 각 layer의 ssm_state 추출.
    Returns: {layer_idx: Tensor(nheads, headdim, d_state)}
    """
    inf = InferenceParams(max_seqlen=input_ids.shape[1], max_batch_size=1)
    with torch.no_grad():
        _ = model(input_ids.to(device), inference_params=inf)
    return {
        k: v[1].squeeze(0).cpu().float()
        for k, v in inf.key_value_memory_dict.items()
    }


# ─── A_disc 계산 ─────────────────────────────────────────────────────────────
def get_A_disc(model, n_layers):
    """각 layer, head의 A_disc (실제 discrete-time decay) 계산.
    Returns: A_disc_all shape (n_layers, n_heads)
    """
    A_disc_all = []
    for i in range(n_layers):
        mixer  = model.backbone.layers[i].mixer
        A_log  = mixer.A_log.detach().cpu()
        dt     = F.softplus(mixer.dt_bias.detach().cpu())
        A_disc = torch.exp(-torch.exp(A_log) * dt)  # (n_heads,)
        A_disc_all.append(A_disc.numpy())
    return np.array(A_disc_all)  # (n_layers, n_heads)


# ─── Head type 분류 ──────────────────────────────────────────────────────────
def classify_heads(A_disc_all, slow_thr=SLOW_THRESHOLD, fast_thr=FAST_THRESHOLD):
    """Type A (slow+high), B (slow+low), C (fast) 분류.
    Returns: type_mask dict with boolean arrays (n_layers, n_heads)
    """
    slow_mask = A_disc_all >= slow_thr
    fast_mask = A_disc_all <  fast_thr
    mid_mask  = ~slow_mask & ~fast_mask
    return {'slow': slow_mask, 'mid': mid_mask, 'fast': fast_mask}


# ─── Head별 rank 계산 ────────────────────────────────────────────────────────
def compute_head_ranks(states, n_layers, n_heads):
    """현재 state에서 (n_layers, n_heads) rank 행렬 반환."""
    ranks = np.zeros((n_layers, n_heads))
    for li in range(n_layers):
        if li not in states:
            continue
        for h in range(n_heads):
            ranks[li, h] = effective_rank(states[li][h])
    return ranks

In [5]:
# ─── 모델별 메인 실험 ─────────────────────────────────────────────────────────
def run_model_experiment(model_name, tokenizer, T_range, n_samples, device):
    """
    단일 모델에 대해 전체 분석 실행.
    Returns: results dict
    """
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")

    # 1. 모델 로드
    model = MambaLMHeadModel.from_pretrained(
        model_name, device=device, dtype=torch.float32
    )
    model.eval()

    mixer      = model.backbone.layers[0].mixer
    cfg        = model.config
    N_LAYERS   = cfg.n_layer
    D_STATE    = mixer.d_state
    HEADDIM    = mixer.headdim
    N_HEADS    = mixer.nheads
    CHUNK_SIZE = mixer.chunk_size
    MAX_RANK   = min(HEADDIM, D_STATE)

    print(f"  n_layers={N_LAYERS}, n_heads={N_HEADS}, "
          f"headdim={HEADDIM}, d_state={D_STATE}, max_rank={MAX_RANK}")

    # 2. A_disc 계산
    A_disc_all = get_A_disc(model, N_LAYERS)  # (n_layers, n_heads)
    head_types = classify_heads(A_disc_all)

    slow_ratio = head_types['slow'].mean()
    fast_ratio = head_types['fast'].mean()
    print(f"  Head type ratio → slow(A): {slow_ratio:.2%}  "
          f"mid(B): {(1-slow_ratio-fast_ratio):.2%}  fast(C): {fast_ratio:.2%}")

    # 3. 데이터 준비
    T_max    = max(T_range)
    passages = get_passages(n_samples, T_max, tokenizer)
    randoms  = get_random_passages(n_samples, T_max)

    # 4. T별 rank trajectory (전체 head mean/max/min + slow-head only)
    def extract_trajectory(passage_list, desc):
        # (n_samples, n_T, n_layers)
        all_mean  = np.zeros((len(passage_list), len(T_range), N_LAYERS))
        all_max   = np.zeros_like(all_mean)
        all_min   = np.zeros_like(all_mean)
        all_slow  = np.zeros_like(all_mean)  # slow head만 mean

        for p_idx, passage in enumerate(tqdm(passage_list, desc=desc)):
            for t_idx, T in enumerate(T_range):
                states = get_ssm_states(model, passage[:, :T], device)
                ranks  = compute_head_ranks(states, N_LAYERS, N_HEADS)
                # ranks: (n_layers, n_heads)

                all_mean[p_idx, t_idx] = ranks.mean(axis=-1)
                all_max[p_idx, t_idx]  = ranks.max(axis=-1)
                all_min[p_idx, t_idx]  = ranks.min(axis=-1)

                # slow head만 평균 (layer별로 slow head가 없으면 nan)
                for li in range(N_LAYERS):
                    slow_mask_li = head_types['slow'][li]  # (n_heads,)
                    if slow_mask_li.sum() > 0:
                        all_slow[p_idx, t_idx, li] = ranks[li][slow_mask_li].mean()
                    else:
                        all_slow[p_idx, t_idx, li] = np.nan

        return {
            'mean'    : all_mean.mean(axis=0).T,   # (n_layers, n_T)
            'max'     : all_max.mean(axis=0).T,
            'min'     : all_min.mean(axis=0).T,
            'slow_mean': np.nanmean(all_slow, axis=0).T,  # (n_layers, n_T)
            'T_range' : T_range,
        }

    wiki_traj = extract_trajectory(passages, f"{model_name.split('/')[-1]} wiki")
    rand_traj = extract_trajectory(randoms,  f"{model_name.split('/')[-1]} rand")

    # 5. T=256에서 scatter data (A_disc vs rank)
    states_256 = get_ssm_states(model, passages[0][:, :256], device)
    ranks_256  = compute_head_ranks(states_256, N_LAYERS, N_HEADS)

    # 6. 결과 정리
    results = {
        'model_name' : model_name,
        'config'     : {
            'n_layers' : N_LAYERS,
            'n_heads'  : N_HEADS,
            'headdim'  : HEADDIM,
            'd_state'  : D_STATE,
            'max_rank' : MAX_RANK,
            'chunk_size': CHUNK_SIZE,
        },
        'A_disc_all' : A_disc_all,    # (n_layers, n_heads)
        'head_types' : head_types,    # slow/mid/fast bool masks
        'ranks_256'  : ranks_256,     # (n_layers, n_heads) at T=256
        'wiki_traj'  : wiki_traj,
        'rand_traj'  : rand_traj,
        'head_ratio' : {
            'slow': float(head_types['slow'].mean()),
            'mid' : float(head_types['mid'].mean()),
            'fast': float(head_types['fast'].mean()),
        },
    }

    # 모델 unload (VRAM 절약)
    del model
    torch.cuda.empty_cache()

    return results

## 4. 전체 모델 실험 실행

모델별로 순차 실행 → VRAM 절약 (모델 로드 후 실험 후 unload).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")
tokenizer.pad_token = tokenizer.eos_token

all_results = {}

for model_name in MODEL_LIST:
    tag = model_name.split('/')[-1]
    save_path = os.path.join(SAVE_DIR, f"{tag}_results.npz")

    # 이미 저장된 결과 있으면 skip
    if os.path.exists(save_path):
        print(f"[skip] {tag} - 결과 파일 존재: {save_path}")
        data = np.load(save_path, allow_pickle=True)
        all_results[tag] = data['results'].item()
        continue

    try:
        results = run_model_experiment(
            model_name, tokenizer, T_RANGE, N_SAMPLES, device
        )
        all_results[tag] = results

        # 저장 (numpy array 포함)
        np.savez(save_path, results=results)
        print(f"[saved] {save_path}")

    except Exception as e:
        print(f"[error] {model_name}: {e}")
        torch.cuda.empty_cache()

print(f"\n완료: {list(all_results.keys())}")


Model: state-spaces/mamba2-130m
  n_layers=24, n_heads=24, headdim=64, d_state=128, max_rank=64
  Head type ratio → slow(A): 24.65%  mid(B): 61.46%  fast(C): 13.89%


mamba2-130m rand: 100%|██████████| 20/20 [06:53<00:00, 20.69s/it]


[saved] ./multi_model_results/mamba2-130m_results.npz

Model: state-spaces/mamba2-370m


config.json:   0%|          | 0.00/331 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/737M [00:00<?, ?B/s]

...

[saved] ./multi_model_results/mamba2-370m_results.npz

Model: state-spaces/mamba2-780m


config.json:   0%|          | 0.00/331 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

  n_layers=48, n_heads=48, headdim=64, d_state=128, max_rank=64
  Head type ratio → slow(A): 24.57%  mid(B): 72.05%  fast(C): 3.39%


mamba2-780m wiki:   0%|          | 0/20 [00:00<?, ?it/s]

## 5. 시각화 함수

In [ ]:
MODEL_COLORS = {
    'mamba2-130m': '#1f77b4',
    'mamba2-370m': '#ff7f0e',
    'mamba2-780m': '#2ca02c',
    'mamba2-1.3b': '#d62728',
    'mamba2-2.7b': '#9467bd',
}


# ─── Plot 1: Head type ratio 비교 (bar chart) ──────────────────────────────
def plot_head_type_ratio(all_results):
    tags    = list(all_results.keys())
    slow_r  = [all_results[t]['head_ratio']['slow'] for t in tags]
    mid_r   = [all_results[t]['head_ratio']['mid']  for t in tags]
    fast_r  = [all_results[t]['head_ratio']['fast'] for t in tags]

    x   = np.arange(len(tags))
    w   = 0.25
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(x - w, slow_r, w, label='Type A (slow, A_disc≥0.99)', color='steelblue')
    ax.bar(x,     mid_r,  w, label='Type B (mid)',                color='orange')
    ax.bar(x + w, fast_r, w, label='Type C (fast, A_disc<0.50)', color='salmon')

    ax.set_xticks(x)
    ax.set_xticklabels(tags, rotation=15)
    ax.set_ylabel('Head ratio')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=9)
    ax.set_title('Head Type Distribution by Model Scale', fontsize=13)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'head_type_ratio.png'), dpi=150)
    plt.show()


# ─── Plot 2: A_disc vs rank scatter per layer (선택 모델) ──────────────────
def plot_scatter_A_disc_vs_rank(results, n_layer_cols=6):
    tag      = results['model_name'].split('/')[-1]
    N_LAYERS = results['config']['n_layers']
    N_HEADS  = results['config']['n_heads']
    A_disc   = results['A_disc_all']   # (n_layers, n_heads)
    ranks    = results['ranks_256']    # (n_layers, n_heads)

    n_rows = (N_LAYERS + n_layer_cols - 1) // n_layer_cols
    fig, axes = plt.subplots(n_rows, n_layer_cols,
                             figsize=(3.2 * n_layer_cols, 3 * n_rows))
    axes = axes.flatten()

    for li in range(N_LAYERS):
        ax = axes[li]
        # Type별 색 구분
        slow = results['head_types']['slow'][li]
        fast = results['head_types']['fast'][li]
        mid  = results['head_types']['mid'][li]

        if slow.sum() > 0:
            ax.scatter(A_disc[li][slow], ranks[li][slow],
                       s=30, color='steelblue',  alpha=0.8, label='A')
        if mid.sum() > 0:
            ax.scatter(A_disc[li][mid],  ranks[li][mid],
                       s=30, color='orange',     alpha=0.8, label='B')
        if fast.sum() > 0:
            ax.scatter(A_disc[li][fast], ranks[li][fast],
                       s=30, color='salmon',     alpha=0.8, label='C')

        ax.set_title(f'L{li}', fontsize=9)
        ax.set_xlim(0, 1)
        ax.set_xlabel('A_disc', fontsize=7)
        ax.set_ylabel('rank',   fontsize=7)
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.3)

    # 빈 subplot 숨기기
    for i in range(N_LAYERS, len(axes)):
        axes[i].set_visible(False)

    handles = [
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='steelblue', label='Type A'),
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='orange',    label='Type B'),
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='salmon',    label='Type C'),
    ]
    fig.legend(handles=handles, loc='lower right', fontsize=9)
    plt.suptitle(f'{tag} — A_disc vs Effective Rank (T=256)', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'{tag}_scatter.png'), dpi=150)
    plt.show()


# ─── Plot 3: Slow head rank saturation curve (모델 비교) ───────────────────
def plot_slow_head_saturation(all_results, layer_fraction=0.5):
    """
    모든 모델의 slow head rank curve를 한 그래프에.
    layer_fraction: 0=first layer, 0.5=middle, 1=last
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, traj_key, label in zip(axes,
                                    ['wiki_traj', 'rand_traj'],
                                    ['Wikitext', 'Random']):
        for tag, results in all_results.items():
            N_LAYERS = results['config']['n_layers']
            li       = int(N_LAYERS * layer_fraction)
            T_pos    = np.array(results[traj_key]['T_range'])
            slow_m   = results[traj_key]['slow_mean'][li]   # (n_T,)
            color    = MODEL_COLORS.get(tag, None)

            if not np.all(np.isnan(slow_m)):
                ax.plot(T_pos, slow_m, 'o-', color=color,
                        lw=2, label=tag, alpha=0.85)

        ax.set_xlabel('Token count (T)', fontsize=11)
        ax.set_ylabel('Effective Rank (slow heads)', fontsize=11)
        ax.set_title(f'{label} — Layer {int(100*layer_fraction)}%', fontsize=12)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    plt.suptitle('Slow Head (Type A) Rank Saturation — Model Comparison',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'slow_head_saturation.png'), dpi=150)
    plt.show()


# ─── Plot 4: Layer별 slow head 비율 heatmap (모델별) ──────────────────────
def plot_slow_head_heatmap(all_results):
    tags = list(all_results.keys())
    fig, axes = plt.subplots(1, len(tags), figsize=(4 * len(tags), 5))
    if len(tags) == 1:
        axes = [axes]

    for ax, tag in zip(axes, tags):
        results  = all_results[tag]
        A_disc   = results['A_disc_all']  # (n_layers, n_heads)
        im = ax.imshow(A_disc, aspect='auto', cmap='RdYlGn',
                       vmin=0, vmax=1, origin='lower')
        plt.colorbar(im, ax=ax, label='A_disc')
        ax.set_xlabel('Head index', fontsize=10)
        ax.set_ylabel('Layer',      fontsize=10)
        ax.set_title(tag, fontsize=11)

    plt.suptitle('A_disc Heatmap (Layer × Head) — All Models',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'A_disc_heatmap.png'), dpi=150)
    plt.show()


# ─── Plot 5: 모델 스케일 vs slow head 비율 + max rank (scaling law 후보) ───
def plot_scaling_summary(all_results):
    tags       = list(all_results.keys())
    param_map  = {
        'mamba2-130m': 0.13, 'mamba2-370m': 0.37,
        'mamba2-780m': 0.78, 'mamba2-1.3b': 1.3, 'mamba2-2.7b': 2.7
    }
    params      = [param_map.get(t, 0) for t in tags]
    slow_ratios = [all_results[t]['head_ratio']['slow'] for t in tags]
    max_ranks   = [
        all_results[t]['ranks_256'][all_results[t]['head_types']['slow']].max()
        if all_results[t]['head_types']['slow'].any() else 0
        for t in tags
    ]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(params, slow_ratios, 'o-', color='steelblue', lw=2, ms=8)
    for p, r, t in zip(params, slow_ratios, tags):
        axes[0].annotate(t, (p, r), textcoords='offset points',
                          xytext=(5, 5), fontsize=8)
    axes[0].set_xlabel('Model size (B params)', fontsize=11)
    axes[0].set_ylabel('Type A head ratio', fontsize=11)
    axes[0].set_title('Slow Head Ratio vs Model Scale', fontsize=12)
    axes[0].grid(alpha=0.3)

    axes[1].plot(params, max_ranks, 's-', color='tomato', lw=2, ms=8)
    for p, r, t in zip(params, max_ranks, tags):
        axes[1].annotate(t, (p, r), textcoords='offset points',
                          xytext=(5, 5), fontsize=8)
    axes[1].set_xlabel('Model size (B params)', fontsize=11)
    axes[1].set_ylabel('Max rank of Type A heads', fontsize=11)
    axes[1].set_title('Max Rank (Type A) vs Model Scale', fontsize=12)
    axes[1].grid(alpha=0.3)

    plt.suptitle('Scaling Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'scaling_summary.png'), dpi=150)
    plt.show()

## 6. 결과 시각화

In [ ]:
# Head type 비율 비교
plot_head_type_ratio(all_results)

In [ ]:
# 각 모델별 A_disc vs rank scatter
for tag, results in all_results.items():
    plot_scatter_A_disc_vs_rank(results)

In [ ]:
# A_disc heatmap (layer × head)
plot_slow_head_heatmap(all_results)

In [ ]:
# Slow head rank saturation 비교 (중간 layer)
plot_slow_head_saturation(all_results, layer_fraction=0.25)
plot_slow_head_saturation(all_results, layer_fraction=0.50)
plot_slow_head_saturation(all_results, layer_fraction=0.75)

In [ ]:
# Scaling law 후보 분석
plot_scaling_summary(all_results)

## 7. 수치 요약 테이블

In [ ]:
print(f"{'Model':<18} | {'n_layers':>8} | {'n_heads':>7} | "
      f"{'max_rank':>8} | {'A(slow)%':>9} | {'B(mid)%':>8} | {'C(fast)%':>9} | "
      f"{'A_disc max':>10} | {'A_disc min':>10}")
print("-" * 110)

for tag, results in all_results.items():
    cfg    = results['config']
    ratio  = results['head_ratio']
    A_disc = results['A_disc_all']
    print(f"{tag:<18} | {cfg['n_layers']:>8} | {cfg['n_heads']:>7} | "
          f"{cfg['max_rank']:>8} | {ratio['slow']:>9.1%} | {ratio['mid']:>8.1%} | "
          f"{ratio['fast']:>9.1%} | {A_disc.max():>10.6f} | {A_disc.min():>10.6f}")

## 8. 결과 JSON 저장

In [ ]:
summary = {}
for tag, results in all_results.items():
    summary[tag] = {
        'config'    : results['config'],
        'head_ratio': results['head_ratio'],
        'A_disc_stats': {
            'max' : float(results['A_disc_all'].max()),
            'min' : float(results['A_disc_all'].min()),
            'mean': float(results['A_disc_all'].mean()),
            'per_layer_slow_count': results['head_types']['slow'].sum(axis=1).tolist(),
        },
        'rank_at_256_stats': {
            'slow_mean': float(results['ranks_256'][results['head_types']['slow']].mean())
                         if results['head_types']['slow'].any() else 0,
            'all_mean' : float(results['ranks_256'].mean()),
            'all_max'  : float(results['ranks_256'].max()),
        },
    }

with open(os.path.join(SAVE_DIR, 'multi_model_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Summary saved: {SAVE_DIR}/multi_model_summary.json")

## 9. 분석 포인트

결과 해석 체크리스트:

| 질문 | 확인 방법 |
|---|---|
| 모델 크기에 따라 Type A head 비율이 증가하는가? | Plot 1, Plot 5 |
| 큰 모델일수록 slow head의 rank가 더 높은가? | Plot 5 |
| Layer별 A_disc 패턴이 모델 크기와 상관없이 유사한가? | Plot 4 heatmap |
| Slow head rank saturation T*가 모델 크기에 따라 달라지는가? | Plot 3 |
| Wiki vs Random 차이가 모델 크기에 따라 달라지는가? | Plot 3 |

**핵심 가설:**
```
큰 모델 = Type A head 비율 높고 rank도 높음
→ "모델 스케일이 long-term memory capacity를 결정"
→ Mamba RAG에서 큰 모델이 더 많은 문서 정보를 state에 압축 가능
```